### This script uses Open-Meteo API to extract the ERA5-Land daily mean temperature (Tmean) using point locations of lsoa coordinates. The script uses the licensed (commercial) version of the API, but the free version can also be used though only limited API calls will be available in that case (read Open-Meteo documentation for further details). Open-Metero includes an option to extract the daily Tmean for the local timezone. To handle time zones robustly, the parameter 'auto' can be set in the API call. However, this does not always work correctly for all coordinates. To bypass the error, the timezone for each coordinate is first set manually, and only when the extraction fails using the manual timezone, the API call reverts to the 'auto' options using a try/exception error catching approach.

### Data:
####  Urban centre coordinates: GHS_UCDB_GLOBE_R2024A (Document last update: 08/09/2026)
####  Coverage: 34,753 lsoas (latitude/longitude points)


### Useful links

#### https://open-meteo.com/en/docs/historical-weather-api#latitude=-34.92,-27.47&longitude=138.6,153.03&start_date=2023-01-01&end_date=2023-12-31&hourly=&daily=temperature_2m_mean&timezone=auto
#### https://stackoverflow.com/questions/40700039/how-can-you-clone-a-conda-environment-into-the-root-environment
#### https://github.com/open-meteo/open-meteo/discussions/650
#### https://github.com/open-meteo/open-meteo/discussions/696
#### https://pypi.org/project/timezonefinder/

In [1]:
import sys
#sys.path.append('/home/lshmm22/anaconda3/envs/open_meteo/lib/python3.8/site-packages')
print(sys.executable) 

C:\Users\lshhc13\AppData\Local\miniconda3\envs\open-meteo\python.exe


In [2]:
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install openmeteo_requests requests_cache retry_requests timezonefinder tzwhere

### Install required packages before loading

In [3]:
import os
import requests
import openmeteo_requests
import requests_cache
import pandas as pd
import pyarrow
from retry_requests import retry
from tzwhere import tzwhere
from datetime import datetime
from timezonefinder import TimezoneFinder
import pytz
#import rpy2.robjects as ro # works but the date column gets messed up when saving as .RDS
#from rpy2.robjects import pandas2ri # same as above

In [4]:
modules = [
    "requests", "openmeteo_requests", "requests_cache", "pandas",
    "pyarrow", "retry_requests", "tzwhere", "timezonefinder", "pytz"
]

for m in modules:
    try:
        __import__(m)
        print(f"✅ {m} imported successfully")
    except ImportError as e:
        print(f"❌ {m} not installed: {e}")


✅ requests imported successfully
✅ openmeteo_requests imported successfully
✅ requests_cache imported successfully
✅ pandas imported successfully
✅ pyarrow imported successfully
✅ retry_requests imported successfully
✅ tzwhere imported successfully
✅ timezonefinder imported successfully
✅ pytz imported successfully


### Set path to home directory, input csv file (ucdb_coordinates), output folder, and output parquet file

In [5]:

print(os.getcwd()) 

C:\Users\lshhc13\Projects\Huiqi\notebooks


In [6]:
from pathlib import Path

# 1) Base project folder under your Windows user profile
base = Path.home() / "Projects" / "Huiqi"

# 2) Input & output paths (do NOT overwrite them later)
input_coords = base / "input_data" / "lsoa_coor.csv"
out_dir_parquet = base / "output_data" / "parquet" / "LSOA_daily_ERA5"
out_file_parquet = out_dir_parquet / "LSOA_daily_ERA5_2026_t2m.parquet"

# 3) Make sure output folders exist
out_dir_parquet.mkdir(parents=True, exist_ok=True)

# 4) Helpful debug prints
print("Looking for:", input_coords)
print("Exists?", input_coords.exists())

# 5) Read
mcc_ghsl_df = pd.read_csv(input_coords)   # Path objects work; str(input_coords) also fine


Looking for: C:\Users\lshhc13\Projects\Huiqi\input_data\lsoa_coor.csv
Exists? True


In [7]:
print("CWD:", os.getcwd())
print("Home:", Path.home())

CWD: C:\Users\lshhc13\Projects\Huiqi\notebooks
Home: C:\Users\lshhc13


### Read the input coordinates from the input .csv file. Rename columns and round the lon, lat to decimals

In [8]:
#mcc_ghsl_df = pd.read_csv(input_mcc_ghsl_coords_file)
mcc_ghsl_df = mcc_ghsl_df[['LSOA11CD', 'lat', 'lon']]
#mcc_ghsl_df.rename(columns={'GCPNT_LAT': 'lat'}, inplace=True)
#mcc_ghsl_df.rename(columns={'GCPNT_LON': 'lon'}, inplace=True)
mcc_ghsl_df["lat"] = mcc_ghsl_df["lat"].astype(float)
mcc_ghsl_df["lon"] = mcc_ghsl_df["lon"].astype(float)
print(mcc_ghsl_df)

        LSOA11CD        lat       lon
0      E01000001  51.518101 -0.097060
1      E01000002  51.518681 -0.091970
2      E01000003  51.521759 -0.095230
3      E01000005  51.514519 -0.076280
4      E01000006  51.538761  0.089318
...          ...        ...       ...
34748  W01001954  51.517300 -3.255810
34749  W01001955  51.621368 -3.942370
34750  W01001956  51.615070 -4.015040
34751  W01001957  51.615669 -3.946440
34752  W01001958  51.616550 -3.933030

[34753 rows x 3 columns]


### Create a column of timezone in the above DF using TimezoneFinder() and a function to apply it row-wise on the DF

In [9]:
tf = TimezoneFinder()

def get_timezone(row):
    return tf.timezone_at(lng=row['lon'], lat=row['lat'])  # Note: longitude first

### Apply function row-wise

In [10]:
mcc_ghsl_df['timezone'] = mcc_ghsl_df.apply(get_timezone, axis=1)
mcc_ghsl_df

,LSOA11CD,lat,lon,timezone
0,E01000001,51.518101,-0.097060,Europe/London
1,E01000002,51.518681,-0.091970,Europe/London
2,E01000003,51.521759,-0.095230,Europe/London
3,E01000005,51.514519,-0.076280,Europe/London
4,E01000006,51.538761,0.089318,Europe/London
...,...,...,...,...
34748,W01001954,51.517300,-3.255810,Europe/London
34749,W01001955,51.621368,-3.942370,Europe/London
34750,W01001956,51.615070,-4.015040,Europe/London
34751,W01001957,51.615669,-3.946440,Europe/London


In [11]:
# Check if any rows in the column 'timezone' have missing values
print(mcc_ghsl_df['timezone'].isnull().any())

# How many are missing
print(mcc_ghsl_df['timezone'].isnull().sum())

#To check which rows are missing
mcc_ghsl_df[mcc_ghsl_df['timezone'].isnull()]

False
0


,LSOA11CD,lat,lon,timezone


### Define a function to extract the daily Tmean for the required time period. Use the API key and set the base_url to "https://customer-archive-api.open-meteo.com/v1/archive"
### Alternatively, set base_url to "https://api.open-meteo.com/v1/forecast" if using the free version.

### Within this parent function, use try and exception to check for errors if the timezone from the DF column returns an error when extracting the daily Tmean. Use the 'auto' option for the timezone in that case.

In [12]:
def fetch_daily_mean_temp(lat, lon, timezone):
    #base_url = "https://api.open-meteo.com/v1/forecast"  # Use this for free version of Open-Meteo and comment the apikey parameter below
    base_url = "https://customer-archive-api.open-meteo.com/v1/archive"
    
    date_ranges = [
        ("2026-05-01", "2026-09-15")#, ("2011-01-01", "2025-12-31")
    ]

    def make_request(tz, start, end):
        params = {
            "latitude": lat,
            "longitude": lon,
            #"start_date": "1990-01-01", # For some reason, the data ends up starting from 1989-12-31
            #"end_date": "2025-12-31",    
            "start_date": start,
            "end_date": end,            
            "daily": "temperature_2m_mean",
            #"models": "era5_land",
            "apikey": os.environ["OPEN_METEO_API_KEY"],
            "timezone": tz
        }
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        if "daily" not in data or "temperature_2m_mean" not in data["daily"]:
            raise ValueError("Missing daily mean temperature in response")
        return data["daily"]
        
    all_times = []
    all_temps = []
    
    for start, end in date_ranges:
        try:
            result = make_request(timezone, start, end)
        except Exception as e:
            print(f"⚠️ Failed with timezone '{timezone}' at ({lat}, {lon}): {e}")
            print("Retrying with timezone='auto'...")
            try:
                result = make_request("auto", start, end)
            except Exception as e2:
                print(f"❌ Failed with 'auto' as well: {e2}")
                return None
        
        # combine 
        all_times.extend(result["time"])
        all_temps.extend(result["temperature_2m_mean"])
    
    return {"time": all_times, "temperature_2m_mean": all_temps}

### Now run the above function for each row (location) in the DF. The result of the extaction will be in a form of list, which can be flattened as individual rows and appended to the output DF. The columns retained are LSOA11CD, lat, lon, date and the temperature. But note that the output DF is not a Pandas DF and the conversion to Pandas DF is done separately in the block below

In [13]:
%%time

flattened_rows = []

for _, row in mcc_ghsl_df.iterrows():
    print(f"\nCurrent city is: {row['LSOA11CD']}\n")
    daily_data = fetch_daily_mean_temp(row['lat'], row['lon'], row['timezone'])
    
    if daily_data:
        for date, temp in zip(daily_data['time'], daily_data['temperature_2m_mean']):
            flattened_rows.append({
                'LSOA11CD': row['LSOA11CD'],
                'lat': row['lat'],
                'lon': row['lon'],
                'date': date,
                # Column renamed to t2m
                't2m': temp
            })
    else:
        print(f"❌ No data for {row['LSOA11CD']}")



Current city is: E01000001


Current city is: E01000002


Current city is: E01000003


Current city is: E01000005


Current city is: E01000006


Current city is: E01000007


Current city is: E01000008


Current city is: E01000009


Current city is: E01000010


Current city is: E01000011


Current city is: E01000012


Current city is: E01000013


Current city is: E01000014


Current city is: E01000015


Current city is: E01000016


Current city is: E01000017


Current city is: E01000018


Current city is: E01000019


Current city is: E01000020


Current city is: E01000021


Current city is: E01000022


Current city is: E01000023


Current city is: E01000024


Current city is: E01000025


Current city is: E01000027


Current city is: E01000028


Current city is: E01000029


Current city is: E01000030


Current city is: E01000031


Current city is: E01000032


Current city is: E01000033


Current city is: E01000034


Current city is: E01000035


Current city is: E01000036


Current city 

### Convert to Pandas DF

In [14]:
mcc_ghsl_df_flat = pd.DataFrame(flattened_rows)
print(mcc_ghsl_df_flat)

          LSOA11CD        lat      lon        date   t2m
0        E01000001  51.518101 -0.09706  2026-05-01  17.6
1        E01000001  51.518101 -0.09706  2026-05-02  16.0
2        E01000001  51.518101 -0.09706  2026-05-03  15.4
3        E01000001  51.518101 -0.09706  2026-05-04  13.9
4        E01000001  51.518101 -0.09706  2026-05-05  13.5
...            ...        ...      ...         ...   ...
4795909  W01001958  51.616550 -3.93303  2026-09-11  15.7
4795910  W01001958  51.616550 -3.93303  2026-09-12  16.6
4795911  W01001958  51.616550 -3.93303  2026-09-13  18.1
4795912  W01001958  51.616550 -3.93303  2026-09-14  18.4
4795913  W01001958  51.616550 -3.93303  2026-09-15  16.5

[4795914 rows x 5 columns]


### Check if any rows in the column 'timezone' have missing values

In [15]:
print(mcc_ghsl_df_flat['t2m'].isnull().any())
# False

# How many are missing
mcc_ghsl_df_flat['t2m'].isnull().sum()

#To check which rows are missing
#df_combined[df_combined['t2m'].isnull()]

False


np.int64(0)

### Check min and max across all daily Tmean (i.e., all timesteps and all locations)

In [16]:
mcc_ghsl_df_flat['t2m'].min()

np.float64(2.3)

In [17]:
mcc_ghsl_df_flat['t2m'].max()

np.float64(30.2)

#### Save the large Pandas DF as a parquet file

In [18]:
mcc_ghsl_df_flat['date'] = pd.to_datetime(mcc_ghsl_df_flat['date'])
mcc_ghsl_df_flat.to_parquet(out_file_parquet, engine="pyarrow", index=False)

#### Read the parquet file back to test everything is okay

In [19]:
%%time
mcc_ghsl_df_flat_loaded = pd.read_parquet(out_file_parquet, engine="pyarrow")
mcc_ghsl_df_flat_loaded

CPU times: total: 438 ms
Wall time: 240 ms


,LSOA11CD,lat,lon,date,t2m
0,E01000001,51.518101,-0.09706,2026-05-01,17.6
1,E01000001,51.518101,-0.09706,2026-05-02,16.0
2,E01000001,51.518101,-0.09706,2026-05-03,15.4
3,E01000001,51.518101,-0.09706,2026-05-04,13.9
4,E01000001,51.518101,-0.09706,2026-05-05,13.5
...,...,...,...,...,...
4795909,W01001958,51.616550,-3.93303,2026-09-11,15.7
4795910,W01001958,51.616550,-3.93303,2026-09-12,16.6
4795911,W01001958,51.616550,-3.93303,2026-09-13,18.1
4795912,W01001958,51.616550,-3.93303,2026-09-14,18.4


In [20]:
mcc_ghsl_df_flat_loaded.dtypes

LSOA11CD               str
lat                float64
lon                float64
date        datetime64[us]
t2m                float64
dtype: object

### End of script